In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"
TABLES_DIR = PROJECT_DIR / "results" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

Project directory: c:\Users\ASUS\Documents\GitHub\Communication-Network-Failure-and-Resilience


**Load the processed files**

In [3]:
edges = pd.read_csv(PROCESSED_DIR / "email_edges_clean.csv")
departments = pd.read_csv(
    PROCESSED_DIR / "email_departments_clean.csv"
)

# Ensure node IDs use the same data type in both files
edges["source"] = pd.to_numeric(edges["source"], errors="raise").astype(int)
edges["target"] = pd.to_numeric(edges["target"], errors="raise").astype(int)
departments["node"] = pd.to_numeric(
    departments["node"], errors="raise"
).astype(int)

department_map = departments.set_index("node")["department"].to_dict()

print("Email connections:", len(edges))
print("Department records:", len(departments))
print("Unique departments:", departments["department"].nunique())

display(edges.head())
display(departments.head())

Email connections: 25571
Department records: 1005
Unique departments: 42


,source,target
0,0,1
1,2,3
2,2,4
3,5,6
4,5,7


,node,department
0,0,1
1,1,1
2,2,21
3,3,21
4,4,21


In [4]:
G = nx.Graph()

G.add_edges_from(
    edges[["source", "target"]].itertuples(
        index=False, name=None
    )
)

G.add_nodes_from(departments["node"])

nx.set_node_attributes(G, department_map, "department")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))

Nodes: 1005
Edges: 16706
Connected components: 20


In [6]:
print("Calculating betweenness centrality...")

betweenness = nx.betweenness_centrality(
    G,
    normalized=True
)

degree = dict(G.degree())

bridge_results = []

for node in G.nodes():
    node_department = department_map.get(node)
    neighbors = list(G.neighbors(node))

    neighbor_departments = [
        department_map.get(neighbor)
        for neighbor in neighbors
    ]

    known_neighbor_departments = [
        d for d in neighbor_departments if pd.notna(d)
    ]

    cross_department_neighbors = sum(
        1 for d in known_neighbor_departments
        if pd.notna(node_department) and d != node_department
    )

    known_department_neighbors = len(known_neighbor_departments)

    cross_department_fraction = (
        cross_department_neighbors / known_department_neighbors
        if known_department_neighbors > 0
        else 0
    )

    bridge_results.append({
        "node": node,
        "department": node_department,
        "degree": degree[node],
        "betweenness_centrality": betweenness[node],
        "cross_department_neighbors": cross_department_neighbors,
        "neighbor_departments": len(set(known_neighbor_departments)),
        "cross_department_fraction": cross_department_fraction
    })

bridge_df = pd.DataFrame(bridge_results)

bridge_df = bridge_df.sort_values(
    "betweenness_centrality",
    ascending=False
).reset_index(drop=True)

display(bridge_df.head(20))

bridge_df.to_csv(
    TABLES_DIR / "department_bridge_centrality.csv",
    index=False
)

print("Saved department_bridge_centrality.csv")

Calculating betweenness centrality...


,node,department,degree,betweenness_centrality,cross_department_neighbors,neighbor_departments,cross_department_fraction
0,160,36,347,0.087415,334,36,0.965318
1,86,36,218,0.037789,202,31,0.930876
2,5,25,171,0.030995,164,36,0.964706
3,82,36,233,0.027881,212,38,0.913793
4,121,36,234,0.027842,215,39,0.922747
5,107,36,221,0.024340,204,37,0.927273
6,13,26,180,0.023565,173,34,0.966480
7,377,7,139,0.023175,99,17,0.717391
8,62,36,216,0.022510,204,36,0.948837
9,64,25,170,0.021924,163,32,0.964497


Saved department_bridge_centrality.csv


In [7]:
cross_edges = edges.copy()

cross_edges["source_department"] = (
    cross_edges["source"].map(department_map)
)
cross_edges["target_department"] = (
    cross_edges["target"].map(department_map)
)

cross_edges = cross_edges.dropna(
    subset=["source_department", "target_department"]
)

cross_edges = cross_edges[
    cross_edges["source_department"]
    != cross_edges["target_department"]
].copy()

print("Total recorded email connections:", len(edges))
print("Cross-department email connections:", len(cross_edges))

display(cross_edges.head(10))

cross_edges.to_csv(
    TABLES_DIR / "cross_department_edges.csv",
    index=False
)

Total recorded email connections: 25571
Cross-department email connections: 16284


,source,target,source_department,target_department
4,5,7,25,14
6,10,11,9,14
7,12,13,14,26
8,12,14,14,4
9,15,16,17,34
31,41,42,5,34
35,49,50,36,37
37,52,53,7,4
39,54,56,22,21
40,54,57,22,21


In [8]:
department_pairs = (
    cross_edges.groupby(
        ["source_department", "target_department"]
    )
    .size()
    .reset_index(name="connection_count")
    .sort_values("connection_count", ascending=False)
)

display(department_pairs.head(20))

department_pairs.to_csv(
    TABLES_DIR / "cross_department_pair_counts.csv",
    index=False
)

,source_department,target_department,connection_count
1040,36,4,229
119,4,5,170
651,21,22,159
148,4,36,155
685,22,21,154
1051,36,15,151
157,5,4,151
115,4,0,150
1050,36,14,136
1037,36,1,128
